In [0]:
from pyspark.sql import *
from pyspark.sql.functions import  *

In [0]:
init_load_flag = int(dbutils.widgets.get('init_load_flag'))

### DATA READING from silver

In [0]:
df = spark.sql('select * from  retail_databricks_catalog.silver.customer_silver')

In [0]:
df.count()

4000

### removing duplicates

In [0]:
df = df.dropDuplicates(subset=['customer_id'])

In [0]:
df.count()

2000

### DIviding new vs old records

In [0]:
if init_load_flag == 0:
    df_old = spark.sql('select dimcustomer, customer_id, create_date, update_date from  retail_databricks_catalog.gold_schema.dimcustomer')
else:
    df_old = spark.sql('select 0 dimcustomer, 0 customer_id, 0 create_date, 0 update_date  from  retail_databricks_catalog.silver.customer_silver where 1 = 0')

In [0]:
df_old.display()

dimcustomer,customer_id,create_date,update_date
1,C00001,2025-07-05T02:25:06.608Z,2025-07-05T02:25:06.608Z
2,C00002,2025-07-05T02:25:06.608Z,2025-07-05T02:25:06.608Z
3,C00003,2025-07-05T02:25:06.608Z,2025-07-05T02:25:06.608Z
4,C00004,2025-07-05T02:25:06.608Z,2025-07-05T02:25:06.608Z
5,C00005,2025-07-05T02:25:06.608Z,2025-07-05T02:25:06.608Z
6,C00006,2025-07-05T02:25:06.608Z,2025-07-05T02:25:06.608Z
7,C00007,2025-07-05T02:25:06.608Z,2025-07-05T02:25:06.608Z
8,C00008,2025-07-05T02:25:06.608Z,2025-07-05T02:25:06.608Z
9,C00009,2025-07-05T02:25:06.608Z,2025-07-05T02:25:06.608Z
10,C00010,2025-07-05T02:25:06.608Z,2025-07-05T02:25:06.608Z


### renaming columns of old records

In [0]:
df_old = df_old.withColumnRenamed('dimcustomer', 'old_dimcustomer')\
    .withColumnRenamed('customer_id', 'old_customer_id')\
    .withColumnRenamed('create_date', 'old_create_date')\
    .withColumnRenamed('update_date', 'old_update_date')


In [0]:
df_old.display()

old_dimcustomer,old_customer_id,old_create_date,old_update_date
1,C00001,2025-07-05T02:25:06.608Z,2025-07-05T02:25:06.608Z
2,C00002,2025-07-05T02:25:06.608Z,2025-07-05T02:25:06.608Z
3,C00003,2025-07-05T02:25:06.608Z,2025-07-05T02:25:06.608Z
4,C00004,2025-07-05T02:25:06.608Z,2025-07-05T02:25:06.608Z
5,C00005,2025-07-05T02:25:06.608Z,2025-07-05T02:25:06.608Z
6,C00006,2025-07-05T02:25:06.608Z,2025-07-05T02:25:06.608Z
7,C00007,2025-07-05T02:25:06.608Z,2025-07-05T02:25:06.608Z
8,C00008,2025-07-05T02:25:06.608Z,2025-07-05T02:25:06.608Z
9,C00009,2025-07-05T02:25:06.608Z,2025-07-05T02:25:06.608Z
10,C00010,2025-07-05T02:25:06.608Z,2025-07-05T02:25:06.608Z


### APPLYING JOIN WITH THE OLD RECORDS

In [0]:
df_joined = df.join(df_old, df.customer_id == df_old.old_customer_id, 'left')
df_joined.display()


customer_id,email,city,state,_rescued_data,domain,fullname,old_dimcustomer,old_customer_id,old_create_date,old_update_date
C00001,rushjeff@ryan.org,Johnsonmouth,MS,null,ryan.org,Emily Mooney,1,C00001,2025-07-05T02:25:06.608Z,2025-07-05T02:25:06.608Z
C00002,mccoykiara@kelly.com,Stephenfort,WY,null,kelly.com,Andrea Sellers,2,C00002,2025-07-05T02:25:06.608Z,2025-07-05T02:25:06.608Z
C00003,rebeccamiller@yahoo.com,South Stephenshire,LA,null,yahoo.com,Craig Hayes,3,C00003,2025-07-05T02:25:06.608Z,2025-07-05T02:25:06.608Z
C00004,lawrence05@campbell.info,Chrisland,ND,null,campbell.info,Bryan Scott,4,C00004,2025-07-05T02:25:06.608Z,2025-07-05T02:25:06.608Z
C00005,carrie45@yahoo.com,East Dennistown,RI,null,yahoo.com,Sean Vasquez,5,C00005,2025-07-05T02:25:06.608Z,2025-07-05T02:25:06.608Z
C00006,traceyramos@gmail.com,North Matthew,IN,null,gmail.com,Kevin Mccarthy,6,C00006,2025-07-05T02:25:06.608Z,2025-07-05T02:25:06.608Z
C00007,scottallen@gmail.com,Joneshaven,VA,null,gmail.com,Amanda Doyle,7,C00007,2025-07-05T02:25:06.608Z,2025-07-05T02:25:06.608Z
C00008,sullivanjeremy@horton-adams.com,South Nathanfurt,CT,null,horton-adams.com,Paul Campos,8,C00008,2025-07-05T02:25:06.608Z,2025-07-05T02:25:06.608Z
C00009,dennis03@yahoo.com,Kimberlyview,MD,null,yahoo.com,Mary Green,9,C00009,2025-07-05T02:25:06.608Z,2025-07-05T02:25:06.608Z
C00010,charles58@murillo.net,West Hector,OK,null,murillo.net,James Myers,10,C00010,2025-07-05T02:25:06.608Z,2025-07-05T02:25:06.608Z


### seperating new vs old records

In [0]:
df_new = df_joined.filter(df_joined['old_dimcustomer'].isNull())

In [0]:
df_new.display()


customer_id,email,city,state,_rescued_data,domain,fullname,old_dimcustomer,old_customer_id,old_create_date,old_update_date


In [0]:
df_old = df_joined.filter(df_joined['old_dimcustomer'].isNotNull())

In [0]:
df_old.display()

customer_id,email,city,state,_rescued_data,domain,fullname,old_dimcustomer,old_customer_id,old_create_date,old_update_date
C00001,rushjeff@ryan.org,Johnsonmouth,MS,null,ryan.org,Emily Mooney,1,C00001,2025-07-05T02:25:06.608Z,2025-07-05T02:25:06.608Z
C00002,mccoykiara@kelly.com,Stephenfort,WY,null,kelly.com,Andrea Sellers,2,C00002,2025-07-05T02:25:06.608Z,2025-07-05T02:25:06.608Z
C00003,rebeccamiller@yahoo.com,South Stephenshire,LA,null,yahoo.com,Craig Hayes,3,C00003,2025-07-05T02:25:06.608Z,2025-07-05T02:25:06.608Z
C00004,lawrence05@campbell.info,Chrisland,ND,null,campbell.info,Bryan Scott,4,C00004,2025-07-05T02:25:06.608Z,2025-07-05T02:25:06.608Z
C00005,carrie45@yahoo.com,East Dennistown,RI,null,yahoo.com,Sean Vasquez,5,C00005,2025-07-05T02:25:06.608Z,2025-07-05T02:25:06.608Z
C00006,traceyramos@gmail.com,North Matthew,IN,null,gmail.com,Kevin Mccarthy,6,C00006,2025-07-05T02:25:06.608Z,2025-07-05T02:25:06.608Z
C00007,scottallen@gmail.com,Joneshaven,VA,null,gmail.com,Amanda Doyle,7,C00007,2025-07-05T02:25:06.608Z,2025-07-05T02:25:06.608Z
C00008,sullivanjeremy@horton-adams.com,South Nathanfurt,CT,null,horton-adams.com,Paul Campos,8,C00008,2025-07-05T02:25:06.608Z,2025-07-05T02:25:06.608Z
C00009,dennis03@yahoo.com,Kimberlyview,MD,null,yahoo.com,Mary Green,9,C00009,2025-07-05T02:25:06.608Z,2025-07-05T02:25:06.608Z
C00010,charles58@murillo.net,West Hector,OK,null,murillo.net,James Myers,10,C00010,2025-07-05T02:25:06.608Z,2025-07-05T02:25:06.608Z


### Preparing DF_old

In [0]:
# dropping all the columns that are not required

df_old = df_old.drop('old_customer_id', 'old_update_date')

# renaming old_dimcustomer dimcustomer
df_old = df_old.withColumnRenamed('old_dimcustomer', 'dimcustomer')

#renaming 'old_create_date' to create_date, 


df_old = df_old.withColumnRenamed('old_create_date', 'create_date')
df_old = df_old.withColumn("create_date", to_timestamp(col("create_date")))

# recreating 'Update_date column with current timestamp

df_old = df_old.withColumn('update_date', current_timestamp()) 

In [0]:
df_old.display()

customer_id,email,city,state,_rescued_data,domain,fullname,dimcustomer,create_date,update_date
C00001,rushjeff@ryan.org,Johnsonmouth,MS,null,ryan.org,Emily Mooney,1,2025-07-05T02:25:06.608Z,2025-07-05T03:14:36.046Z
C00002,mccoykiara@kelly.com,Stephenfort,WY,null,kelly.com,Andrea Sellers,2,2025-07-05T02:25:06.608Z,2025-07-05T03:14:36.046Z
C00003,rebeccamiller@yahoo.com,South Stephenshire,LA,null,yahoo.com,Craig Hayes,3,2025-07-05T02:25:06.608Z,2025-07-05T03:14:36.046Z
C00004,lawrence05@campbell.info,Chrisland,ND,null,campbell.info,Bryan Scott,4,2025-07-05T02:25:06.608Z,2025-07-05T03:14:36.046Z
C00005,carrie45@yahoo.com,East Dennistown,RI,null,yahoo.com,Sean Vasquez,5,2025-07-05T02:25:06.608Z,2025-07-05T03:14:36.046Z
C00006,traceyramos@gmail.com,North Matthew,IN,null,gmail.com,Kevin Mccarthy,6,2025-07-05T02:25:06.608Z,2025-07-05T03:14:36.046Z
C00007,scottallen@gmail.com,Joneshaven,VA,null,gmail.com,Amanda Doyle,7,2025-07-05T02:25:06.608Z,2025-07-05T03:14:36.046Z
C00008,sullivanjeremy@horton-adams.com,South Nathanfurt,CT,null,horton-adams.com,Paul Campos,8,2025-07-05T02:25:06.608Z,2025-07-05T03:14:36.046Z
C00009,dennis03@yahoo.com,Kimberlyview,MD,null,yahoo.com,Mary Green,9,2025-07-05T02:25:06.608Z,2025-07-05T03:14:36.046Z
C00010,charles58@murillo.net,West Hector,OK,null,murillo.net,James Myers,10,2025-07-05T02:25:06.608Z,2025-07-05T03:14:36.046Z


### PREPARING DF_NEW

In [0]:
# dropping all the columns that are not required

df_new = df_new.drop('old_dimcustomer', 'old_customer_id', 'old_update_date', 'old_create_date')





In [0]:
df_new.display()


customer_id,email,city,state,_rescued_data,domain,fullname


### Surrogate key from 1

In [0]:
df_new = df_new.withColumn('dimcustomer', monotonically_increasing_id()+lit(1))


In [0]:
df_new.limit(10).display()

customer_id,email,city,state,_rescued_data,domain,fullname,dimcustomer


In [0]:
# recreating 'Update_date, current_date  column with current timestamp

df_new = df_new.withColumn('update_date', current_timestamp()) 
df_new = df_new.withColumn('create_date', current_timestamp()) 

In [0]:
df_new.limit(10).display()

customer_id,email,city,state,_rescued_data,domain,fullname,dimcustomer,update_date,create_date


### ADDING MAX SURROGATE KEY

In [0]:
if init_load_flag == 1:
  max_surrogate_key = 0
else:
  df_maxsur = spark.sql("select max(dimcustomer) as max_surrogate_key from retail_databricks_catalog.gold_schema.dimcustomer")
  ### converting max_surrogatekey into maxsurrogatekey variable
  max_surrogate_key = df_maxsur.collect()[0]['max_surrogate_key']

In [0]:
df_new.limit(1).display()

customer_id,email,city,state,_rescued_data,domain,fullname,dimcustomer,update_date,create_date


In [0]:
df_new = df_new.withColumn('dimcustomer', lit(max_surrogate_key)+col('dimcustomer'))

### UNION OF DF_OLD AND DF_NEW

In [0]:
df_old.printSchema()


root
 |-- customer_id: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- domain: string (nullable = true)
 |-- fullname: string (nullable = true)
 |-- dimcustomer: long (nullable = true)
 |-- create_date: timestamp (nullable = true)
 |-- update_date: timestamp (nullable = false)



In [0]:
df_new.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- domain: string (nullable = true)
 |-- fullname: string (nullable = true)
 |-- dimcustomer: long (nullable = false)
 |-- update_date: timestamp (nullable = false)
 |-- create_date: timestamp (nullable = false)



In [0]:
df_final = df_new.unionByName(df_old)

In [0]:
df_final.display()


customer_id,email,city,state,_rescued_data,domain,fullname,dimcustomer,update_date,create_date
C00001,rushjeff@ryan.org,Johnsonmouth,MS,null,ryan.org,Emily Mooney,1,2025-07-05T03:16:35.615Z,2025-07-05T02:25:06.608Z
C00002,mccoykiara@kelly.com,Stephenfort,WY,null,kelly.com,Andrea Sellers,2,2025-07-05T03:16:35.615Z,2025-07-05T02:25:06.608Z
C00003,rebeccamiller@yahoo.com,South Stephenshire,LA,null,yahoo.com,Craig Hayes,3,2025-07-05T03:16:35.615Z,2025-07-05T02:25:06.608Z
C00004,lawrence05@campbell.info,Chrisland,ND,null,campbell.info,Bryan Scott,4,2025-07-05T03:16:35.615Z,2025-07-05T02:25:06.608Z
C00005,carrie45@yahoo.com,East Dennistown,RI,null,yahoo.com,Sean Vasquez,5,2025-07-05T03:16:35.615Z,2025-07-05T02:25:06.608Z
C00006,traceyramos@gmail.com,North Matthew,IN,null,gmail.com,Kevin Mccarthy,6,2025-07-05T03:16:35.615Z,2025-07-05T02:25:06.608Z
C00007,scottallen@gmail.com,Joneshaven,VA,null,gmail.com,Amanda Doyle,7,2025-07-05T03:16:35.615Z,2025-07-05T02:25:06.608Z
C00008,sullivanjeremy@horton-adams.com,South Nathanfurt,CT,null,horton-adams.com,Paul Campos,8,2025-07-05T03:16:35.615Z,2025-07-05T02:25:06.608Z
C00009,dennis03@yahoo.com,Kimberlyview,MD,null,yahoo.com,Mary Green,9,2025-07-05T03:16:35.615Z,2025-07-05T02:25:06.608Z
C00010,charles58@murillo.net,West Hector,OK,null,murillo.net,James Myers,10,2025-07-05T03:16:35.615Z,2025-07-05T02:25:06.608Z


### SCD type 1

In [0]:
from delta.tables import *


In [0]:
if (spark.catalog.tableExists("retail_databricks_catalog.gold_schema.dimcustomer")):

    dlt_obj = DeltaTable.forPath(spark, 'abfss://gold@databricksretailproject.dfs.core.windows.net/dimcustomer')

    dlt_obj.alias('trg').merge(df_final.alias('src'), 'trg.dimcustomer = src.dimcustomer')\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()
else:
    df_final.write.mode('overwrite')\
    .format('delta')\
    .option('path', 'abfss://gold@databricksretailproject.dfs.core.windows.net/dimcustomer')\
    .saveAsTable('retail_databricks_catalog.gold_schema.dimcustomer')   

        


In [0]:
        %sql
select * from retail_databricks_catalog.gold_schema.dimcustomer

customer_id,email,city,state,_rescued_data,domain,fullname,dimcustomer,update_date,create_date
C00001,rushjeff@ryan.org,Johnsonmouth,MS,null,ryan.org,Emily Mooney,1,2025-07-05T02:25:06.608Z,2025-07-05T02:25:06.608Z
C00002,mccoykiara@kelly.com,Stephenfort,WY,null,kelly.com,Andrea Sellers,2,2025-07-05T02:25:06.608Z,2025-07-05T02:25:06.608Z
C00003,rebeccamiller@yahoo.com,South Stephenshire,LA,null,yahoo.com,Craig Hayes,3,2025-07-05T02:25:06.608Z,2025-07-05T02:25:06.608Z
C00004,lawrence05@campbell.info,Chrisland,ND,null,campbell.info,Bryan Scott,4,2025-07-05T02:25:06.608Z,2025-07-05T02:25:06.608Z
C00005,carrie45@yahoo.com,East Dennistown,RI,null,yahoo.com,Sean Vasquez,5,2025-07-05T02:25:06.608Z,2025-07-05T02:25:06.608Z
C00006,traceyramos@gmail.com,North Matthew,IN,null,gmail.com,Kevin Mccarthy,6,2025-07-05T02:25:06.608Z,2025-07-05T02:25:06.608Z
C00007,scottallen@gmail.com,Joneshaven,VA,null,gmail.com,Amanda Doyle,7,2025-07-05T02:25:06.608Z,2025-07-05T02:25:06.608Z
C00008,sullivanjeremy@horton-adams.com,South Nathanfurt,CT,null,horton-adams.com,Paul Campos,8,2025-07-05T02:25:06.608Z,2025-07-05T02:25:06.608Z
C00009,dennis03@yahoo.com,Kimberlyview,MD,null,yahoo.com,Mary Green,9,2025-07-05T02:25:06.608Z,2025-07-05T02:25:06.608Z
C00010,charles58@murillo.net,West Hector,OK,null,murillo.net,James Myers,10,2025-07-05T02:25:06.608Z,2025-07-05T02:25:06.608Z
